# Lab 1 — worked solutions

**LDSCI6253 Information Presentation & Data Visualisation** · Northeastern University London

Task 4, done in five stages: **raw → table → clean → visualise → insights**.
Every cell below is the answer key — run it yourself so the outputs sit under the code when you submit.

**The data.** `tfl-daily-cycle-hires.xlsx` sits beside this notebook: total daily hires of
the Santander Cycle Hire scheme, 30 July 2010 to 31 August 2026.
Contains public sector information licensed under the
[Open Government Licence v2.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/2/)
— Transport for London, via the [London Datastore](https://data.london.gov.uk/dataset/number-of-bicycle-hires-2r84d).

**Two traps before you start:** `sheet_name=0` is the *Metadata* sheet, and pointing
`read_excel` at the Datastore URL returns **HTTP 403**. Download the file; read it from disk.

**Alternatives** (same five stages): [London Fire Brigade incidents](https://data.london.gov.uk/dataset/london-fire-brigade-incident-records-em8xy/),
[MPS recorded crime by borough](https://data.london.gov.uk/dataset/mps-recorded-crime-geographic-breakdown-exy3m).
Browse more at [data.london.gov.uk](https://data.london.gov.uk) — filter by CSV.


## Stage 1 · Raw

Look at the file before believing anything about it.


In [ ]:
import pandas as pd

# Download tfl-daily-cycle-hires.xlsx and put it beside this notebook.
# Pointing pandas at the London Datastore URL returns HTTP 403.
path = "tfl-daily-cycle-hires.xlsx"

print(pd.ExcelFile(path).sheet_names)
# ['Metadata', 'Data']   <- sheet 0 is NOT the data

raw = pd.read_excel(path, sheet_name="Data", header=None)
print(raw.shape)          # (5883, 16) — day / month / year blocks side by side
print(raw.iloc[:7, 1:3])  # notes and totals sit above the series


Three things that output tells us, none of which were guessable:

1. The workbook has **two sheets**, and `Data` is the second. `sheet_name=0` would have read twenty-nine rows of description.
2. The sheet is **far wider than a daily series needs**, because day, month and year blocks sit side by side.
3. There is a notes paragraph and **grand totals above the header**, so the header is on row 6, not row 1.


## Stage 2 · Managed table

Still messy — but now we can *see* the rows we will keep. This is the table you manage before cleaning.


In [ ]:
# Peek at the block that will become the clean table (after the five preamble rows).
preview = pd.read_excel(path, sheet_name="Data", skiprows=5, usecols=[1, 2],
                        names=["date", "hires"])
print(preview.head(10).to_string(index=False))
print("...")
print(preview.tail(5).to_string(index=False))
print("rows so far (includes blank trailer):", len(preview))


## Stage 3 · Clean

One row per day, a real date, a real number. Every argument is a decision you should defend.


In [ ]:
df = pd.read_excel(path, sheet_name="Data", skiprows=5,
                   usecols=[1, 2], names=["date", "hires"])

df = df.dropna()
df["date"] = pd.to_datetime(df["date"])
df["hires"] = pd.to_numeric(df["hires"])

# Check against the total the file states about itself:
print(df.head())
print(len(df), int(df["hires"].sum()))
# Expect ~5877 rows and Grand Total 154053134


The `print` is the habit worth stealing. The file states its own grand total, so a clean
read can be checked against it. If your sum does not match, your cleaning is wrong.


## Stage 4 · Visualise

Several charts, each answering one question. Titles state the **finding**, not the subject.


In [ ]:
import matplotlib.pyplot as plt

monthly = df.set_index("date")["hires"].resample("ME").sum()
# Drop part-years that invent cliffs (scheme opened 30 Jul 2010; file may end mid-year)
monthly_full = monthly[(monthly.index.year >= 2011) & (monthly.index.year <= 2025)]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly_full.index, monthly_full.values, linewidth=1.2)
ax.set_title("Hires peak every summer — and 2020 broke the pattern")
ax.set_ylabel("Hires per month")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Seasonality: mean daily hires by calendar month
by_month = df.groupby(df["date"].dt.month)["hires"].mean()
months = ["J","F","M","A","M","J","J","A","S","O","N","D"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 13), by_month.values, color="#C8102E")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_title("July averages about twice December — a hard annual cycle")
ax.set_ylabel("Mean daily hires")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Weekday vs weekend — commuting or leisure?
wd = df.groupby(df["date"].dt.dayofweek < 5)["hires"].mean()
labels = ["Weekend", "Weekday"]
vals = [wd[False], wd[True]]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(labels, vals, color=["#6B7280", "#C8102E"])
ax.set_title("Weekdays outpace weekends — this is commuting infrastructure")
ax.set_ylabel("Mean daily hires")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# COVID shock: April 2019 vs April 2020
apr = {y: int(df[(df["date"].dt.year == y) & (df["date"].dt.month == 4)]["hires"].sum())
       for y in (2019, 2020)}

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["April 2019", "April 2020"], [apr[2019], apr[2020]], color=["#6B7280", "#C8102E"])
ax.set_title("April 2020 fell 34% below April 2019 — first lockdown")
ax.set_ylabel("Hires in the month")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


## Stage 5 · Insights

Compute them. Do not eyeball them. A chart with no sentence under it is half a submission.


In [ ]:
by_month = df.groupby(df["date"].dt.month)["hires"].mean()
print("July mean daily:     %d" % round(by_month[7]))
print("December mean daily: %d" % round(by_month[12]))
print("summer:winter        %.1fx" % (by_month[7] / by_month[12]))

apr = {y: df[(df["date"].dt.year == y) & (df["date"].dt.month == 4)]["hires"].sum()
       for y in (2019, 2020)}
print("April 2019 / 2020:   %d / %d  (%+.0f%%)"
      % (apr[2019], apr[2020], (apr[2020] / apr[2019] - 1) * 100))

annual = df.groupby(df["date"].dt.year)["hires"].sum()
print("2022 / 2023:         %.2fM / %.2fM  (%+.0f%%)"
      % (annual[2022] / 1e6, annual[2023] / 1e6,
         (annual[2023] / annual[2022] - 1) * 100))

weekday = df.groupby(df["date"].dt.dayofweek < 5)["hires"].mean()
print("weekend / weekday:   %d / %d" % (round(weekday[False]), round(weekday[True])))


In [ ]:
# Two traps that look like findings
monthly = df.set_index("date")["hires"].resample("ME").sum()
print("lowest month in the file:", monthly.idxmin().strftime("%B %Y"),
      int(monthly.min()), "<- two days of trading: the scheme opened on 30 July")
after = monthly[monthly.index.year >= 2012]
print("lowest month from 2012:  ", after.idxmin().strftime("%B %Y"),
      int(after.min()), "<- a seasonal floor AND a lockdown, at once")


### The paragraph this is all for

> Santander Cycle hires follow a hard annual cycle: July averages **34,212** hires a day
> against December's **17,132**, a factor of **two**. Two events break the pattern. April
> 2020 fell **34%** below April 2019 (591,294 against 890,148) — the first lockdown — and
> annual hires dropped **26%** between 2022 and 2023, from 11.51M to 8.53M. The second is
> the more interesting, because **this dataset cannot explain it**: it records hires and
> nothing else, so fares, dock changes, e-bikes and weather are all outside it. Naming the
> gap is the finding; inventing a cause would not be. The series also contradicts the
> obvious guess about who uses it — weekdays average **27,500** hires a day against
> weekends' **22,998**, so this is commuting infrastructure, not a leisure amenity.

### Two traps

- **Lowest month in the file is July 2010** — not a collapse; two days of trading after opening on the 30th.
- **Lowest month from 2012 is January 2021** — seasonal floor *and* lockdown. Do not attribute it to either alone.

### What to submit

1. Your notebook **with outputs visible** (run every cell, save, then upload).
2. A short description: what the data is, where it came from, what you did.
3. Task 2 critique on Canvas with the original URL.

On Canvas by the **Monday after the lab, 23:59**.
